### **Цей ноутбук буде фінальним, правильним рішенням, на мою думку, враховуючи минулі досліди та виявлення помилок**

### Основні помилки, які я виявив:
- Неправильний KFold для регресії, бо я хоч і подумав, що відсортований dataset, і параметр shuffle=false будуть запобігати загляданню в майбутнє, але забув, що ми можемо навчатись на майбутніх, а тестувати на минулих. (2-3-4-5 -> тренування, 1->тест)
- Не виконана калібровка ймовірностей в класифікації. Так наш фінальний розрахунок може бути більш оптимістичний, тому вводимо Калібровку ймовірностей.
- Також маємо фіксований поріг класифікації(базовий 0.5), хоча в той ж час маємо великий дисбаланс класів, тому треба змінювати поріг.
- Підібрані гарні гіперпараметри в LightGBM порівняно з базовим CatBoost працюють гірше.
- Використання GridSearchCV буде дуже довго виконуватись, тому краще використати RandomizedSearchCV
------------------

In [ ]:
# Libraries
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, KFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score, classification_report, PrecisionRecallDisplay, average_precision_score
from lightgbm import LGBMClassifier, LGBMRegressor

In [ ]:
# Зчитуємо дані та оглядаємо
actions = pd.read_csv("../../data/task_2_users_actions.csv")
params = pd.read_csv("../../data/task_2_users_params.csv")

------------------
# EDA

In [ ]:
df_actions = actions.copy()
df_params = params.copy()

In [ ]:
print("task_2_users_actions.csv: ")
print("Shape: ", df_actions.shape)
display(df_actions.head(3))

print("task_2_users_params.csv: ")
print("Shape: ", df_params.shape)
display(df_params.head(3))

In [ ]:
# Детальна інформація по датасетам
df_actions.info(memory_usage='deep')
df_params.info(memory_usage='deep')

In [ ]:
plt.figure(figsize=[8, 5])
sns.boxplot(y=params['revenue_7d'])
plt.title('Розподіл revenue_7d (з викидами)')
plt.show()

In [ ]:
miss_data_in_params = df_params.isnull().sum()
miss_data_in_actions = df_actions.isnull().sum()

print("Miss data(users_actions):\n", miss_data_in_actions, '\n')
print("Miss data(users_params):\n", miss_data_in_params, '\n')

print("Params shape:", df_params.shape)
print("Actions shape:", df_actions.shape)

print("Unique users in params:", df_params['id_user'].nunique())
print("Unique users in actions:", df_actions['id_user'].nunique())

In [ ]:
display(df_params[['age', 'revenue_7d']].describe())
display(df_actions[['sum_payments', 'cnt_payments', 'sum_credits_spend']].describe())

In [ ]:
print(df_params['gender'].value_counts())
print(df_params['device_platform'].value_counts())

In [ ]:
# Таргет — сильно скошений, більшість = 0
print('Total lines:', len(params))
print('revenue_7d == 0:', (params['revenue_7d'] == 0).sum())
print('revenue_7d > 0:', (params['revenue_7d'] > 0).sum())
print()
print(params['revenue_7d'].describe())

Велика кількість нулів, тому єдина регресійна модель на всьому датасеті буде тягнути прогноз до нуля.

In [ ]:
# Топ країни за середньою виручкою
country_revenue = (
    params.groupby('country')['revenue_7d']
    .agg(['mean', 'sum', 'count'])
    .sort_values('mean', ascending=False)
)
country_revenue.head(15)

Країни з дуже малою кількістю користувачів (`count` в одиницях) дають нестабільне `mean` (наприклад, один випадковий великий платіж підніме середнє до топу). У фінальному висновку варто явно фільтрувати за мінімальним `count` (наприклад, `count >= 100`) перед тим, як називати "топ країни"

In [ ]:
MIN_USERS = 100
top_countries = (
    country_revenue[country_revenue['count'] >= MIN_USERS]
    .sort_values('mean', ascending=False)
    .head(5)
)
top_countries

In [ ]:
# Revenue по traffic_type
params.groupby('traffic_type_id')['revenue_7d'].agg(['mean', 'sum', 'count'])

In [ ]:
# Revenue по device_platform
params.groupby('device_platform')['revenue_7d'].agg(['mean', 'sum', 'count']).sort_values('mean', ascending=False)

In [ ]:
# Розподіл виручки тільки серед тих, хто заплатив
payers = params[params['revenue_7d'] > 0]['revenue_7d']

plt.title('Розподіл виручки у платників')
plt.xlabel('Revenue ($)')
plt.ylabel('Кількість юзерів (log)')
plt.hist(payers, bins=50, color='darkorange', edgecolor='white', log=True)

plt.tight_layout()
plt.show()

print(f'Медіана виручки у платників: ${payers.median():.2f}')
print(f'75-й перцентиль: ${payers.quantile(0.75):.2f}')
print(f'95-й перцентиль: ${payers.quantile(0.95):.2f}')
print(f'99-й перцентиль: ${payers.quantile(0.99):.2f}')
print(f'Максимум: ${payers.max():.2f}')


-------------------
# Feature Engineering

In [ ]:
# Переведемо часові фічі в потрібний нам формат
df_params['timestamp_reg'] = pd.to_datetime(df_params['timestamp_reg'], format='mixed')

df_actions['timestamp_interval_start'] = pd.to_datetime(df_actions['timestamp_interval_start'], format='mixed')

df_actions['timestamp_interval_end'] = pd.to_datetime(df_actions['timestamp_interval_end'], format='mixed')

In [ ]:
# Видалення дублікатів.
print("Duplicate: ", df_params.duplicated().sum())
print("Duplicate: ", df_actions.duplicated().sum())

df_actions = df_actions.drop_duplicates()
df_params = df_params.sort_values('timestamp_reg').drop_duplicates(subset='id_user', keep='first')

print("After delete duplicate: ", df_params.duplicated().sum())
print("After delete duplicate: ", df_actions.duplicated().sum())
print("Now lines in params:", df_params.shape[0])

In [ ]:
df_actions.head(8)

In [ ]:
# Агрегуємо actions по юзеру
# Кожен юзер має 4 інтервали (6-годинні) => агрегуємо суми та розраховуємо поведінкові фічі

# ВИПРАВЛЕНО: раніше act_active_intervals рахувався лише по cnt_visit_other_users (перегляди
# чужих профілів). Тепер інтервал вважається активним, якщо в ньому була будь-яка дія: платіж, витрата кредитів, повернення або перегляд.

activity_cols = ['sum_payments', 'cnt_payments', 'sum_credits_spend', 'cnt_returns', 'cnt_visit_other_users']
df_actions['has_activity'] = (df_actions[activity_cols].sum(axis=1) > 0).astype(int)

actions_agg = df_actions.groupby('id_user').agg(
    act_sum_payments=('sum_payments', 'sum'),
    act_max_payments=('sum_payments', 'max'),
    act_cnt_payments=('cnt_payments', 'sum'),
    act_sum_credits=('sum_credits_spend', 'sum'),
    act_cnt_returns=('cnt_returns', 'sum'),
    act_sum_visits=('cnt_visit_other_users', 'sum'),
    act_max_visits=('cnt_visit_other_users', 'max'),
    act_active_intervals=('has_activity', 'sum'),
).reset_index()

print(actions_agg.shape)
actions_agg.head()

In [ ]:
# З'єднуємо дві таблиці по ключу "id_user". (LEFT JOIN)
df = df_params.merge(actions_agg, on='id_user', how='left')

-----------------
# Підготовка для моделювання. Обробка фінального датасету

In [2]:
print('Shape:', df.shape, '\n')
print('Miss data:\n', df.isnull().sum(), '\n')
print('Duplicate: ', df.duplicated().sum())
display(df.head(3))

NameError: name 'df' is not defined

In [ ]:
# Створюємо нові фічі, які пов'язані з часом. Початкову колонку видаляємо
df['reg_hour'] = df['timestamp_reg'].dt.hour
df['reg_dayofweek'] = df['timestamp_reg'].dt.dayofweek
df['reg_is_weekend'] = (df['reg_dayofweek'] >= 5).astype(int)
df = df.drop('timestamp_reg', axis=1) # В нас вже масив відсортований, тому ця колонка не несе ніякої користі. Перед розбиттям на вибірки, нам не можна використовувати SHUFFLE.

In [ ]:
str_col = df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
df[str_col] = df[str_col].fillna('unknown').astype(str)
cat_features = str_col
print("Категоріальні фічі для CatBoost:", cat_features)

In [ ]:
df.info(memory_usage='deep')

In [ ]:
num_cols = df.select_dtypes(include=['number'])
corr = num_cols.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(
    corr,
    cmap='coolwarm',
    vmin=-1, vmax=1,
    center=0,
    annot=True,
    fmt='.2f',
    square=True,
    linewidths=.5,
    cbar_kws={"shrink": .7}
)

plt.title('Матриця кореляцій числових ознак', fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Перевіряємо на константні колонки (не несуть користі для моделі)
const_cols = [c for c in num_cols.columns if df[c].nunique() <= 1]
print("Константні числові колонки (кандидати на видалення):", const_cols)

--------------------
# Оцінка даних, вибір моделі та стратегії


In [ ]:
# Відсоток тих, хто нічого не купив (False) і тих, хто заплатив (True)
payer_percentage = (df['revenue_7d'] > 0).value_counts(normalize=True) * 100
print("Відсоток платників та неплатників:\n", payer_percentage)

**Проблема нульової виручки:** близько 98% користувачів мають `revenue_7d = 0`, тому єдина регресійна модель на всьому датасеті зміщуватиме прогнози до нуля, ігноруючи цільовий (платний) сегмент.

**Двохетапний підхід:**
1. **Класифікатор (CatBoost):** оцінює ймовірність конверсії користувача в платника.
2. **Регресор (CatBoost):** навчається виключно на сегменті платників, прогнозує суму.

------------------------
# Навчання моделей


In [ ]:
X = df.drop(columns=['id_user', 'revenue_7d'] + const_cols)
y = df['revenue_7d']

# shuffle=False зберігає хронологію (дані відсортовані за timestamp_reg)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
y_bin_train = y_train.gt(0).astype(int)
y_bin_test = y_test.gt(0).astype(int)

print(f"Тренувальна вибірка: {X_train.shape[0]} рядків")
print(f"Тестова вибірка: {X_test.shape[0]} рядків")
print(f"Платників у train: {y_bin_train.sum()} ({y_bin_train.mean()*100:.1f}%)")
print(f"Платників у test: {y_bin_test.sum()}  ({y_bin_test.mean()*100:.1f}%)")

cat_features_in_X = [c for c in cat_features if c in X.columns]
tscv = TimeSeriesSplit(n_splits=3)